## **Setup**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -qU transformers datasets bitsandbytes wandb

In [ ]:
%%capture
# Login to WandB
!wandb login your_wandb_api_key

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import os
import torch
from IPython.display import clear_output
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

from unsloth import (
    FastLanguageModel,
    is_bfloat16_supported,
    UnslothTrainer,
    UnslothTrainingArguments,
)

clear_output()

## **Configs**

In [ ]:
MAX_SEQ_LENGTH = 2048
MODEL_ID = "unsloth/Llama-3.2-3B-bnb-4bit"
# MODEL_ID = "unsloth/Llama-3.2-1B-bnb-4bit"

# TOKENIZER_ID = 'Naod-Demissie/Llama-3.2-3B-bnb-4bit-amh-merged-178k'
# TOKENIZER_ID = 'Naod-Demissie/Llama-3.2-3B-bnb-4bit-amh-50k'
TOKENIZER_ID = "Naod-Demissie/Llama-3.2-3B-bnb-4bit-amh-128k"

DATA_PATH = "/content/drive/MyDrive/Local-LLM/data/LMTextData_normalized_unique_rm_unkfidels_rm_leng1_2.txt"

os.environ["WANDB_PROJECT"] = "Local-LLM-Llama-3.2-3B"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

## **Loading Dataset, Model and Tokenizer**

In [ ]:
# for i in range(len(tokenizer)):
#     print(i, tokenizer.decode(i))

# for i in range(len(amh_tokenizer)):
#     print(i, amh_tokenizer.decode(i))

# _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


# for name, param in model.named_parameters():
#   print(f"{name}: {param.dtype}")

# for name, param in model.named_parameters():
#   print(f"{name}: {param.device}")

# for name, param in model.named_parameters():
#   print(f"Layer: {name}, Trainable: {param.requires_grad}")

In [ ]:
model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
del _

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

In [ ]:
amh_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
len(amh_tokenizer)

tokenizer_config.json:   0%|          | 0.00/51.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

128256

## **Resize Embedding**

In [ ]:
# check the embedding size before resizing
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaExtendedRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Llam

In [ ]:
model.resize_token_embeddings(len(amh_tokenizer))

Embedding(50000, 3072)

In [ ]:
# check the embedding size after resizing
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(50000, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaExtendedRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps

In [ ]:
amh_tokenizer.tokenize("Hello, world!")

['H', 'e', 'l', 'l', 'o', ',', 'Ġ', 'w', 'o', 'r', 'l', 'd', '!']

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

## **Change Special Tokens**

In [ ]:
# check for id of special tokens
model.config

LlamaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "unsloth/Llama-3.2-3B-bnb-4bit",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 24,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandbytes"
  },
  "rms_norm_

In [ ]:
# Change the special tokens in the model config to match the new tokenizer
model.config.bos_token_id = amh_tokenizer.bos_token_id
model.config.eos_token_id = amh_tokenizer.eos_token_id
model.config.pad_token_id = amh_tokenizer.pad_token_id

In [ ]:
# check for id of special tokens after change
model.config

LlamaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "unsloth/Llama-3.2-3B-bnb-4bit",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "eos_token_id": 1,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 24,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "pad_token_id": 4,
  "pretraining_tp": 1,
  "quantization_config": {
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandbytes"
  },
  "rms_norm_eps": 1e-05,
  

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): ModulesToSaveWrapper(
          (original_module): Embedding(50000, 3072)
          (modules_to_save): ModuleDict(
            (default): Embedding(50000, 3072)
          )
        )
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
        

In [ ]:
# Load the data
data = load_dataset("text", data_files=DATA_PATH, split="train")
data

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=128,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "embed_tokens",
        "lm_head",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
    loftq_config=None,
)
clear_output()

In [ ]:
data = data.select(range(500000))

In [ ]:
EOS_TOKEN = amh_tokenizer.eos_token


def formatting_prompts_func(examples):
    return {"text": [example + EOS_TOKEN for example in examples["text"]]}


data = data.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

In [ ]:
for row in data[:5]["text"]:
    print("=========================")
    print(row)

In [ ]:
# from trl import SFTTrainer
# from transformers import TrainingArguments
# from unsloth import is_bfloat16_supported

# trainer = SFTTrainer(
#     model = model,
#     tokenizer = amh_tokenizer,
#     train_dataset = data,
#     dataset_text_field = "text",
#     max_seq_length = 2048,
#     dataset_num_proc = 2,
#     packing = True, # Can make training 5x faster for short sequences.
#     args = TrainingArguments(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 5,
#         max_steps = 60,
#         learning_rate = 2e-4,
#         fp16 = not is_bfloat16_supported(),
#         bf16 = is_bfloat16_supported(),
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         output_dir = "outputs",
#     ),
# )

Generating train split: 0 examples [00:00, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [ ]:
trainer = UnslothTrainer(
    model=model,
    tokenizer=amh_tokenizer,
    train_dataset=data,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=8,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=16,
        warmup_ratio=0.1,
        num_train_epochs=5,
        learning_rate=5e-4,
        embedding_learning_rate=5e-3,
        # learning_rate=5e-5,
        # embedding_learning_rate=5e-6,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.00,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="wandb",
        save_steps=50000,
        # run_name = "local-llm-llama-3.2-1B-bnb-4bit"
        run_name="local-llm-llama-3.2-3B-bnb-4bit-128k",
    ),
)

Map (num_proc=8):   0%|          | 0/500000 [00:00<?, ? examples/s]

## **Train Model**

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.748 GB.
2.846 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 500,000 | Num Epochs = 5
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 16
\        /    Total batch size = 32 | Total steps = 78,125
 "-____-"     Number of trainable parameters = 982,515,712


Step,Training Loss
1,12.683200
2,13.214900
3,13.167200
4,12.963800
5,12.814000
6,12.865300
7,12.695600
8,12.907800
9,12.935000
10,12.790300


Step,Training Loss
1,12.683200
2,13.214900
3,13.167200
4,12.963800
5,12.814000
6,12.865300
7,12.695600
8,12.907800
9,12.935000
10,12.790300


KeyboardInterrupt: 

## **Infer Model Results**

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

462.7198 seconds used for training.
7.71 minutes used for training.
Peak reserved memory = 7.922 GB.
Peak reserved memory for training = 1.938 GB.
Peak reserved memory % of max memory = 53.716 %.
Peak reserved memory for training % of max memory = 13.141 %.


In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference
inputs = tokenizer(["ሆኖም በሽታው ከበረታ በኋላ ህክምና የሚጀምሩ"], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>ሆኖም በሽታው ከበረታ በኋላ ህክምና የሚጀምሩ በሰሜን አፍሪካ በሰሜን አፍሪካ በሰሜን አ�']